# GeoLink: tìm siêu tham số

## 1. Chuẩn bị môi trường


In [ ]:
!git clone https://github.com/tranminhduc9/well-log-imputation.git /kaggle/working/well-log-imputation
%cd /kaggle/working/well-log-imputation


In [ ]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%pip install -q -r requirements.txt


## 2. Đường dẫn và thiết lập chung

Kết quả lưu trong `/kaggle/working/well-log-tuning/`. `search_trials.csv` ghi một dòng cho mỗi lượt thử gồm mô hình, số lượt và RMSE validation trung bình. Cột độ lệch chuẩn chỉ có khi chạy nhiều seed. Tra cấu hình của một lượt qua thứ tự `ParameterGrid` trong `search_protocol.json`, bắt đầu từ 1.

`top3_validation_rmse.csv` lưu đường RMSE của tối đa ba cấu hình tốt nhất mỗi mô hình trong lưới. Điểm xếp hạng và cấu hình JSON nằm ở dòng đầu của mỗi cặp mô hình và hạng. Vẽ `validation_rmse` theo `step`, tách đường theo `rank`, `seed` và `log` nếu có. Mạng nơ-ron dùng epoch và RMSE trung bình qua bốn kịch bản. XGBoost dùng vòng boosting và RMSE Block-20 theo từng log. LOCF không có lưới tìm kiếm hoặc đường huấn luyện.

Các lượt thử chạy trong bộ nhớ. Notebook lưu thêm `best_model_settings.json` và `search_protocol.json` để dùng lại cấu hình.

In [ ]:
from argparse import Namespace
from pathlib import Path
import json
import logging
from zipfile import ZIP_DEFLATED, ZipFile
import sys

import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from sklearn.model_selection import ParameterGrid



In [ ]:
PROJECT_ROOT = Path('/kaggle/working/well-log-imputation')
DATA_ROOT = Path('/kaggle/input/datasets/ductranminh9/geo-link')
OUTPUT_BASE = Path('/kaggle/working/well-log-tuning')
DEVICE = 'cuda'
SEEDS = (129,)
BATCH_SIZE = 32
NEURAL_EPOCHS = 100
NEURAL_PATIENCE = 10
SCENARIOS = ['Single', 'Block-20', 'Block-100', 'Entire-Log']

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runner import run_experiment
from src.experiments.utils import save_json

In [ ]:
OUTPUT_DIR = OUTPUT_BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', force=True)
print('Output:', OUTPUT_DIR)

## 3. Không gian tìm kiếm

Mỗi cấu hình chạy với tất cả seeds. Conv-SAITS dùng multi-scale nên chỉ tìm `multi_scale_kernels`; `kernel_size` dành cho chế độ một kernel và không tham gia tìm kiếm này.


In [7]:
SEARCH_SPACES = {
    'xgboost': {
        'max_depth': [2, 3, 4, 5, 6],
        'learning_rate': [0.01, 0.03, 0.05, 0.1],
        'n_estimators': list(range(100, 1001, 100)),
    },
    'saits': {
        'd_model': [64, 128, 256],
        'learning_rate': [5e-4, 1e-3],
    },
    'flexlognet': {
        'encoder_channels': [(8, 16, 32, 64), (16, 32, 64, 128), (32, 64, 128, 256)],
        'learning_rate': [5e-4, 1e-3],
    },
    'conv_saits': {
        'd_model': [64, 128, 256],
        'learning_rate': [5e-4, 1e-3],
        'encoder_channels': [4, 8],
        'multi_scale_kernels': [(3, 5, 7), (3, 7, 15)],
    },
}
FIXED_SETTINGS = {
    'locf': {'batch_size': BATCH_SIZE},
    'xgboost': {'batch_size': BATCH_SIZE},
    'saits': {
        'batch_size': BATCH_SIZE, 'd_inner': 128, 'n_layers': 2, 'n_heads': 4,
        'ort_weight': 1.0, 'mit_weight': 1.0, 'mit_reduction': 'segment', 'min_delta': 0.0,
    },
    'flexlognet': {
        'batch_size': BATCH_SIZE, 'resample_factors': (4, 8, 16), 'dropout': 0.1,
        'ssim_weight': 1.0, 'statistics_weight': 0.5, 'mean_weight': 4.0,
        'gradient_clip': 1.0, 'min_delta': 0.0,
    },
    'conv_saits': {
        'batch_size': BATCH_SIZE, 'd_inner': 128, 'n_layers': 2, 'n_heads': 4,
        'conv_expansion': 2, 'multi_scale_conv': True, 'depth_encoding': False,
        'masking_strategy': 'mixed', 'mit_reduction': 'segment',
        'ort_weight': 1.0, 'mit_weight': 1.0, 'min_delta': 0.0,
        'masking_rate': 0.2, 'independent_shuffle': True,
        'gap_aware_gate': False, 'decoder': 'shared',
    },
}
GRIDS = {name: list(ParameterGrid(space)) for name, space in SEARCH_SPACES.items()}
search_size = pd.DataFrame({
    'configurations': {name: len(grid) for name, grid in GRIDS.items()},
})
search_size['fits'] = search_size['configurations'] * len(SEEDS)
display(search_size)
save_json(OUTPUT_DIR / 'search_protocol.json', {
    'search_spaces': SEARCH_SPACES, 'fixed_settings': FIXED_SETTINGS,
    'seeds': SEEDS, 'epochs': NEURAL_EPOCHS, 'patience': NEURAL_PATIENCE,
    'data_root': str(DATA_ROOT), 'device': DEVICE, 'batch_size': BATCH_SIZE,
    'evaluate_test': False,
    'trial_index': '1-based order from sklearn.model_selection.ParameterGrid per model',
    'top_k': 3,
    'curve_monitor': {'xgboost': 'Block-20 per log', 'neural': 'mean across four scenarios'},
    'selection': 'Mean validation RMSE across scenarios and seeds',
})


,configurations,fits
xgboost,200,200
saits,6,6
flexlognet,6,6
conv_saits,24,24


## 4. Finetune mô hình


In [ ]:
TRAINING_ARGS = dict(
    data_root=str(DATA_ROOT), seeds=SEEDS, device=DEVICE, evaluate_test=False,
    # brits_epochs=NEURAL_EPOCHS, brits_patience=NEURAL_PATIENCE,
    saits_epochs=NEURAL_EPOCHS, saits_patience=NEURAL_PATIENCE,
    flexlognet_epochs=NEURAL_EPOCHS, flexlognet_patience=NEURAL_PATIENCE,
    conv_saits_epochs=NEURAL_EPOCHS, conv_saits_patience=NEURAL_PATIENCE,
)


def validation_score(metrics, name):
    scores = metrics.query(
        "model == @name and split == 'val' and level == 'overall' and units == 'normalized'"
    ).pivot(index='seed', columns='scenario', values='rmse')
    scores = scores.loc[list(SEEDS), SCENARIOS]
    assert np.isfinite(scores.to_numpy()).all()
    per_seed = scores.mean(axis=1)
    return per_seed.mean(), per_seed.std(ddof=1)


## 5. Tìm cấu hình tốt nhất trên validation

In [9]:
def export_top_curves(top_trials):
    frames = []
    for name, entries in top_trials.items():
        for rank, (score, trial, settings, history) in enumerate(entries, 1):
            points = history.reindex(columns=['seed', 'epoch', 'iteration', 'log', 'validation_rmse'])
            frame = pd.DataFrame({
                'model': name, 'rank': rank, 'trial': trial, 'seed': points['seed'],
                'step_type': 'iteration' if name == 'xgboost' else 'epoch',
                'step': points['epoch'].fillna(points['iteration']).astype('Int64'),
                'log': points['log'], 'validation_rmse': points['validation_rmse'],
                'validation_rmse_mean': np.nan, 'settings': '',
            })
            first = frame.index[0]
            frame.at[first, 'validation_rmse_mean'] = score
            frame.at[first, 'settings'] = json.dumps(settings, separators=(',', ':'), sort_keys=True)
            frames.append(frame)
    pd.concat(frames, ignore_index=True).to_csv(
        OUTPUT_DIR / 'top3_validation_rmse.csv', index=False, float_format='%.6f',
    )


trial_rows = []
top_trials = {}
best_settings = {'locf': FIXED_SETTINGS['locf'].copy()}

for name, grid in GRIDS.items():
    top = top_trials[name] = []
    for trial, parameters in enumerate(grid, 1):
        settings = {name: {**FIXED_SETTINGS[name], **parameters}}
        args = Namespace(
            **TRAINING_ARGS, models=(name,), output_dir=str(OUTPUT_DIR),
            xgb_estimators=parameters.get('n_estimators', 300),
        )
        print(f'{name}: {trial}/{len(grid)} | {parameters}', flush=True)
        trial_metrics, history = run_experiment(
            args, PROJECT_ROOT, settings, save_outputs=False, return_history=True,
        )
        score, score_std = validation_score(trial_metrics, name)
        if history.empty:
            raise ValueError(f'Missing training history for {name} trial {trial}.')
        row = {'model': name, 'trial': trial, 'validation_rmse_mean': score}
        if len(SEEDS) > 1:
            row['validation_rmse_std'] = score_std
        trial_rows.append(row)
        pd.DataFrame(trial_rows).to_csv(
            OUTPUT_DIR / 'search_trials.csv', index=False, float_format='%.6f',
        )

        if len(top) < 3 or (score, trial) < top[-1][:2]:
            top.append((score, trial, settings[name], history))
            top.sort(key=lambda entry: entry[:2])
            del top[3:]
            export_top_curves(top_trials)

    best_score, _, best_settings[name], _ = top[0]
    save_json(OUTPUT_DIR / 'best_model_settings.json', best_settings)
    print(f'{name}: best validation RMSE = {best_score:.6f}')

trials = pd.DataFrame(trial_rows)
best_trials = trials.sort_values('validation_rmse_mean', kind='stable').drop_duplicates('model')
display(best_trials.dropna(axis='columns', how='all'))
display(pd.DataFrame(best_settings).T)

,model,trial,validation_rmse_mean,parameters
212,conv_saits,1,0.275816,"{""d_model"": 64, ""encoder_channels"": 4, ""learni..."
204,saits,5,0.283077,"{""d_model"": 256, ""learning_rate"": 0.0005}"
208,flexlognet,3,0.327299,"{""encoder_channels"": [16, 32, 64, 128], ""learn..."
101,xgboost,102,0.581963,"{""learning_rate"": 0.05, ""max_depth"": 2, ""n_est..."


,batch_size,learning_rate,max_depth,n_estimators,d_inner,n_layers,n_heads,ort_weight,mit_weight,mit_reduction,...,encoder_channels,conv_expansion,multi_scale_conv,depth_encoding,masking_strategy,masking_rate,independent_shuffle,gap_aware_gate,decoder,multi_scale_kernels
locf,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
xgboost,32.0,0.05,2.0,200.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
saits,32,0.0005,NaN,NaN,128,2,4,1.0,1.0,segment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flexlognet,32,0.0005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"(16, 32, 64, 128)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
conv_saits,32,0.0005,NaN,NaN,128,2,4,1.0,1.0,segment,...,4,2,True,False,mixed,0.2,True,False,shared,"(3, 5, 7)"


## 6. Tải kết quả

Tệp ZIP gồm bảng lượt thử, đường RMSE top 3, cấu hình tốt nhất và protocol. Cấu hình được xếp theo RMSE validation trung bình qua các kịch bản và seed. Khi điểm bằng nhau, lượt thử chạy trước được ưu tiên.

In [ ]:
result_files = ['search_trials.csv', 'top3_validation_rmse.csv', 'best_model_settings.json', 'search_protocol.json']
archive_path = OUTPUT_DIR.parent / 'well-log-tuning.zip'
with ZipFile(archive_path, 'w', compression=ZIP_DEFLATED) as archive:
    for name in result_files:
        archive.write(OUTPUT_DIR / name, arcname=name)
display(FileLink(str(archive_path)))
